# Fill in the blank with a tiny BERT

BERT learns language by guessing words that were hidden from it, using the words on both sides
of each gap. Here you build the pieces in NumPy: token and position embeddings, one layer of
attention in both directions, BERT's masking recipe, the masked-token loss and the prediction at
a `[MASK]`. At the end you train the output layer for 600 steps and watch the model
fill in "the cat sat on the `[MASK]`".

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

sentences = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat saw the big dog",
    "the dog saw the big cat",
    "a cat ate the red fish",
    "a dog ate the red bone",
    "the cat ran to the mat",
    "the dog ran to the rug",
]
specials = ["[PAD]", "[CLS]", "[SEP]", "[MASK]"]
vocab = specials + sorted({w for s in sentences for w in s.split()})
idx = {w: i for i, w in enumerate(vocab)}
PAD, CLS, SEP, MASK = 0, 1, 2, 3
n_vocab = len(vocab)


def to_ids(sentence):
    """Token ids with [CLS] in front and [SEP] at the end. "[MASK]" may appear as a word."""
    return np.array([CLS] + [idx[w] for w in sentence.split()] + [SEP])


def to_words(ids):
    return " ".join(vocab[i] for i in ids)


def mask_at(ids, positions):
    """Hide the given positions with [MASK], for testing. Returns (inputs, labels)."""
    inputs, labels = ids.copy(), np.full_like(ids, -100)
    for p in positions:
        labels[p] = ids[p]
        inputs[p] = MASK
    return inputs, labels


def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


ids = np.stack([to_ids(s) for s in sentences])   # 8 sentences, 8 tokens each
T = ids.shape[1]                                  # tokens per sentence, [CLS] and [SEP] included
d = 16                                            # width of every vector inside the model

# Random starting weights. Nothing has been learned yet.
E = 0.5 * rng.normal(size=(n_vocab, d))          # token embeddings, one row per vocabulary entry
P = 0.5 * rng.normal(size=(T, d))                # position embeddings, one row per position
W_q, W_k, W_v = (rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(3))
W_out = rng.normal(size=(d, n_vocab)) / np.sqrt(d)   # output layer: one score per vocabulary entry
b_out = np.zeros(n_vocab)

print(n_vocab, "entries in the vocabulary:", vocab)
print("sentence 0 as ids:", ids[0])
print("and back to words:", to_words(ids[0]))

## Part 1 (coding task): Token and position embeddings

A Transformer does not read words. It reads one vector per token. The table `E` holds one
row of `d` numbers for every entry in the vocabulary, and `P` holds one row for every
position. The vector for a token is its row of `E` plus the row of `P` for the position it
sits at. The position part is what lets the model tell "the" at position 1 from "the" at
position 5, because attention on its own has no sense of order.

Write `embed(ids)`. It takes one sentence as an array of `T` ids and returns a `T x d`
array whose row `t` is `E[ids[t]] + P[t]`. `E[ids]` picks all the rows at once.

In [ ]:
def embed(ids):
    """One vector per token: its token embedding plus the embedding of its position. Shape (T, d)."""
    # TODO
    raise NotImplementedError

In [ ]:
X = embed(ids[0])
assert X.shape == (T, d), f"expected shape {(T, d)}, got {X.shape}"
assert np.allclose(X[3], E[ids[0, 3]] + P[3]), "row 3 should be E[the id at position 3] + P[3]"
assert ids[0, 1] == ids[0, 5] == idx["the"]
assert not np.allclose(X[1], X[5]), "the same word at two positions should get two different vectors"
assert np.allclose(X[1] - X[5], P[1] - P[5]), "those two vectors should differ by exactly P[1] - P[5]"
X1 = embed(ids[1])
assert np.allclose(X[0], X1[0]), "[CLS] at position 0 should get the same vector in every sentence"
assert not np.allclose(X[2], X1[2]), "position 2 holds cat in sentence 0 and dog in sentence 1"
print("Part 1 passed")

## Part 2 (coding task): One layer of attention in both directions

BERT is an encoder, so every token attends to every other token, before it and after it.
One layer of attention works like this. Each row of `X` is turned into a query, a key and a
value: `Q = X @ W_q`, `K = X @ W_k`, `V = X @ W_v`. The score of query `t` against key `s`
is their dot product divided by `sqrt(d)`, so `Q @ K.T / np.sqrt(d)` is a `T x T` matrix
with one row per query position. A softmax along each row turns the scores into weights
that sum to 1, and position `t` then gathers `weights[t] @ V`, a mix of every token's
value. There is no causal mask here. That is the difference from GPT.

Write `self_attention(X)`. It returns two things: `H = X + weights @ V`, which keeps each
token's own vector and adds what it gathered, and the `T x T` matrix `weights`.

In [ ]:
def self_attention(X):
    """One layer of attention in both directions.
    Returns (H, weights): H is X plus what every position gathered, shape (T, d),
    and weights is the T x T matrix of attention weights, one row per query position."""
    # TODO
    raise NotImplementedError

In [ ]:
X = embed(ids[0])
H, A = self_attention(X)
assert H.shape == (T, d), f"H should be {(T, d)}, got {H.shape}"
assert A.shape == (T, T), f"weights should be {(T, T)}, got {A.shape}"
assert np.allclose(A.sum(axis=1), 1), "each row of weights should sum to 1"
assert (A > 0).all(), "no weight should be 0: there is no causal mask, so every position sees every other"
Q, K, V = X @ W_q, X @ W_k, X @ W_v
S = Q @ K.T / np.sqrt(d)
A_ref = np.exp(S - S.max(axis=1, keepdims=True))
A_ref /= A_ref.sum(axis=1, keepdims=True)
assert np.allclose(A, A_ref), "weights should be the row softmax of Q K^T / sqrt(d)"
assert np.allclose(H, X + A_ref @ V), "H should be X plus weights @ V"
# Both directions: change the last word and the output at position 1 moves.
other = ids[0].copy()
other[6] = idx["rug"]
H_other, _ = self_attention(embed(other))
assert not np.allclose(H_other[1], H[1]), "changing the word at position 6 should change the output at position 1"
print("weights at position 1, one per token:", np.round(A[1], 3))
print("Part 2 passed")

## Part 3 (coding task): BERT's masking recipe

Now hide some tokens. BERT picks 15% of the word pieces at random. (The paper says it
masks 15% of *all* WordPiece tokens, but here we skip `[CLS]`, `[SEP]` and `[PAD]`, because
guessing them would teach nothing.) A chosen position keeps its original id in `labels`, and every other position
gets the label -100, which means "no loss here". The input the model sees is changed at the
chosen positions: 80% of them become `[MASK]`, 10% become a random real word, and 10% are
left as they are. The model must recover the original word at every chosen position,
including the ones it can still see.

Write `mask_tokens(ids, rng, rate=0.15)` for one sentence. Draw one number per position
with `rng.random(len(ids))`. A position is chosen when it holds a real word (id 4 or more)
and its number is below `rate`. Then draw a second number per position to decide between
`[MASK]` (below 0.8), a random real word from `rng.integers(4, n_vocab)` (0.8 to 0.9), and
no change. Return `(inputs, labels)`, two arrays with the shape of `ids`. Do not change
`ids` itself.

In [ ]:
def mask_tokens(ids, rng, rate=0.15):
    """BERT's recipe on one sentence. Returns (inputs, labels): inputs is what the model sees,
    labels holds the original id at every chosen position and -100 everywhere else."""
    # TODO
    raise NotImplementedError

In [ ]:
check_rng = np.random.default_rng(1)
ids_copy = ids.copy()
n_real = n_chosen = n_mask = n_random = n_same = 0
for trial in range(3000):
    s = ids[trial % len(ids)]
    inputs, labels = mask_tokens(s, check_rng)
    assert inputs.shape == s.shape and labels.shape == s.shape, "inputs and labels have the shape of ids"
    chosen = labels != -100
    assert not chosen[0] and not chosen[-1], "[CLS] and [SEP] must never be chosen"
    assert (labels[chosen] == s[chosen]).all(), "a chosen position's label is its original id"
    assert (inputs[~chosen] == s[~chosen]).all(), "a position that was not chosen keeps its token"
    assert (inputs[chosen] >= 3).all(), "a chosen position shows [MASK] or a real word"
    n_real += T - 2
    n_chosen += chosen.sum()
    n_mask += (chosen & (inputs == MASK)).sum()
    n_same += (chosen & (inputs == s)).sum()
    n_random += (chosen & (inputs != MASK) & (inputs != s)).sum()
assert (ids == ids_copy).all(), "do not change ids in place"
rate = n_chosen / n_real
share_mask, share_random, share_same = n_mask / n_chosen, n_random / n_chosen, n_same / n_chosen
assert 0.12 < rate < 0.18, f"about 15% of real words should be chosen, got {rate:.3f}"
assert 0.74 < share_mask < 0.86, f"about 80% of chosen positions should show [MASK], got {share_mask:.3f}"
assert 0.05 < share_random < 0.15, f"about 10% of chosen positions should show a random word, got {share_random:.3f}"
assert 0.05 < share_same < 0.16, f"about 10% of chosen positions should be left unchanged, got {share_same:.3f}"
print(f"chosen {rate:.3f} of real words. Of those: [MASK] {share_mask:.2f}, random word {share_random:.2f}, unchanged {share_same:.2f}")
for seed in range(100):
    inputs, labels = mask_tokens(ids[0], np.random.default_rng(seed))
    if (labels != -100).sum() >= 2:
        break
print("original:", to_words(ids[0]))
print("input:   ", to_words(inputs))
print("labels:  ", labels)
print("Part 3 passed")

## Part 4 (coding task): The masked-token loss

At every position the output layer turns the model's vector into one score per vocabulary
entry: `logits = H @ W_out + b_out`, shape `T x n_vocab`. A softmax along a row turns its
scores into probabilities. The loss at a chosen position is minus the log of the
probability given to the true id, and the masked language modeling loss is the mean of
that over the chosen positions only. Positions with label -100 do not count. In the
lesson, a position whose probabilities are mat 0.6, floor 0.3, dog 0.1 costs -ln 0.6,
about 0.511.

Write `mlm_loss(logits, labels)`. Compute the log of the softmax of each row as `logits`
minus the row max minus the log of the sum of `exp(logits - row max)`. Subtracting the max
first keeps `exp` from overflowing. Pick the rows whose label is not -100, read off the
log probability of each true id, and return minus the mean.

In [ ]:
def mlm_loss(logits, labels):
    """Mean cross-entropy over the rows whose label is not -100.
    logits has one row of n_vocab scores per position, labels one id (or -100) per position."""
    # TODO
    raise NotImplementedError

In [ ]:
loss_one = mlm_loss(np.log([[0.6, 0.3, 0.1]]), np.array([0]))
assert abs(loss_one - 0.5108) < 1e-3, f"mat 0.6 should cost -ln 0.6 = 0.511, got {loss_one:.4f}"
logits3 = np.array([[2.0, 0.5, -1.0], [0.1, 0.2, 0.3], [-0.5, 1.5, 0.0]])
loss3 = mlm_loss(logits3, np.array([0, -100, 1]))
assert abs(loss3 - 0.2739) < 1e-3, f"the lesson's three rows should give 0.274, got {loss3:.4f}"
logits3b = logits3.copy()
logits3b[1] = [50.0, -50.0, 0.0]
assert np.isclose(mlm_loss(logits3b, np.array([0, -100, 1])), loss3), "a row with label -100 must not change the loss"
assert abs(mlm_loss(np.zeros((4, n_vocab)), np.array([5, -100, 7, 9])) - np.log(n_vocab)) < 1e-9, "equal scores should cost ln(n_vocab)"
assert np.isfinite(mlm_loss(np.array([[1000.0, 0.0]]), np.array([1]))), "subtract the row max before calling exp"
inputs, labels = mask_at(ids[0], [6])
H, _ = self_attention(embed(inputs))
loss = mlm_loss(H @ W_out + b_out, labels)
print(f"untrained loss at the [MASK] in '{to_words(inputs)}': {loss:.3f}")
print(f"a model that spreads its probability evenly would score ln {n_vocab} = {np.log(n_vocab):.3f}")
assert 0.1 < loss < 15, f"an untrained loss should be near ln(n_vocab), got {loss:.3f}"
print("Part 4 passed")

## Part 5 (coding task): The prediction at a [MASK]

To fill in a blank, take the row of logits at a `[MASK]` position, turn it into
probabilities with a softmax, and read off the most likely entries. `np.argsort(p)[::-1]`
lists the ids from the most likely down.

Write `fill_in(inputs, W_out, b_out, top=3)`. Run the model on `inputs`: `embed`, then
`self_attention`, then `H @ W_out + b_out`. For every position where `inputs` holds
`[MASK]`, collect the `top` most likely words and their probabilities, most likely first.
Return a dictionary from position to that list of `(word, probability)` pairs. The output
layer is passed in because Part 6 trains a new one.

In [ ]:
def fill_in(inputs, W_out, b_out, top=3):
    """For every [MASK] in inputs, the `top` most likely words and their probabilities.
    Returns {position: [(word, probability), ...]}, most likely first."""
    # TODO
    raise NotImplementedError

In [ ]:
inputs, _ = mask_at(ids[0], [2, 6])
guesses = fill_in(inputs, W_out, b_out)
assert set(guesses) == {2, 6}, f"the keys should be the [MASK] positions, got {sorted(guesses)}"
H, _ = self_attention(embed(inputs))
probs = softmax(H @ W_out + b_out, axis=1)
for t, best in guesses.items():
    assert len(best) == 3, "three guesses per position"
    top_words, top_probs = zip(*best)
    assert all(w in vocab for w in top_words), "each guess is a word from the vocabulary"
    assert list(top_probs) == sorted(top_probs, reverse=True), "most likely first"
    assert top_words[0] == vocab[np.argmax(probs[t])], f"the first guess at position {t} should be the most likely word"
    assert abs(top_probs[0] - probs[t].max()) < 1e-9, "a probability should come from the softmax of the logits"
    assert sum(top_probs) <= 1 + 1e-9
# An output layer that ignores the model and always answers mat 0.6, rug 0.3, dog 0.1.
p_fixed = np.full(n_vocab, 1e-12)
p_fixed[idx["mat"]], p_fixed[idx["rug"]], p_fixed[idx["dog"]] = 0.6, 0.3, 0.1
fixed = fill_in(inputs, np.zeros((d, n_vocab)), np.log(p_fixed))[6]
assert [w for w, _ in fixed] == ["mat", "rug", "dog"], f"expected mat, rug, dog, got {fixed}"
assert abs(fixed[0][1] - 0.6) < 1e-6
print("untrained guesses for:", to_words(inputs))
for t, best in guesses.items():
    print(f"  position {t}:", ", ".join(f"{w} {p:.2f}" for w, p in best))
print("Part 5 passed")

## Part 6 (coding task): Train the output layer and fill in the blank

A real BERT trains every weight by backpropagation and draws new masks with the recipe
from Part 3 at every step. Here only the output layer learns, which keeps the gradient to
a few lines, and the embeddings and the attention stay random. For a softmax classifier,
the gradient of the loss with respect to a chosen row's logits is
`softmax(logits[t]) - onehot(labels[t])`, divided by the number of chosen rows because
the loss is a mean. Rows with label -100 get a gradient of zero. Since
`logits = H @ W_out + b_out`, the gradient for `W_out` is `H.T @ dlogits` and the gradient
for `b_out` is the sum of the rows of `dlogits`.

Write `head_gradient(H, labels, W_out, b_out)`. It returns `(dW, db)` with the shapes of
`W_out` and `b_out`. The check compares it with finite differences, then runs 600 steps
of gradient descent. Eight sentences are too few for the random recipe to settle in that
time, so the training batch hides every real word of every sentence once instead, 48
inputs with one `[MASK]` each. Then the trained layer fills in blanks.

In [ ]:
def head_gradient(H, labels, W_out, b_out):
    """Gradient of mlm_loss(H @ W_out + b_out, labels) with respect to W_out and b_out."""
    # TODO
    raise NotImplementedError

In [ ]:
inputs, labels = mask_at(ids[0], [2, 6])
H, _ = self_attention(embed(inputs))
dW, db = head_gradient(H, labels, W_out, b_out)
assert dW.shape == W_out.shape and db.shape == b_out.shape, "return (dW, db) with the shapes of W_out and b_out"
eps = 1e-6
for i, j in [(0, 0), (5, 7), (15, n_vocab - 1)]:
    W_plus, W_minus = W_out.copy(), W_out.copy()
    W_plus[i, j] += eps
    W_minus[i, j] -= eps
    numeric = (mlm_loss(H @ W_plus + b_out, labels) - mlm_loss(H @ W_minus + b_out, labels)) / (2 * eps)
    assert abs(numeric - dW[i, j]) < 1e-5, f"dW[{i}, {j}] should be {numeric:.6f}, got {dW[i, j]:.6f}"
b_plus, b_minus = b_out.copy(), b_out.copy()
b_plus[4] += eps
b_minus[4] -= eps
numeric = (mlm_loss(H @ W_out + b_plus, labels) - mlm_loss(H @ W_out + b_minus, labels)) / (2 * eps)
assert abs(numeric - db[4]) < 1e-5, f"db[4] should be {numeric:.6f}, got {db[4]:.6f}"

# One fixed training batch: every real word of every sentence hidden once.
batch_H, batch_labels = [], []
for s in ids:
    for p in range(1, T - 1):
        inp, lab = mask_at(s, [p])
        batch_H.append(self_attention(embed(inp))[0])
        batch_labels.append(lab)
H_all, labels_all = np.concatenate(batch_H), np.concatenate(batch_labels)
print(len(batch_H), "masked inputs,", (labels_all != -100).sum(), "positions with a loss")

# Gradient descent on the output layer.
W, b = W_out.copy(), b_out.copy()
losses = []
for step in range(600):
    losses.append(mlm_loss(H_all @ W + b, labels_all))
    dW, db = head_gradient(H_all, labels_all, W, b)
    W -= 1.0 * dW
    b -= 1.0 * db
plt.plot(losses)
plt.axhline(np.log(n_vocab), color="gray", linestyle="--", label="ln(n_vocab)")
plt.xlabel("step")
plt.ylabel("masked-token loss")
plt.legend()
plt.show()
print(f"loss {losses[0]:.3f} -> {losses[-1]:.3f}")
assert losses[-1] < 0.6, f"after 600 steps the loss should be under 0.6, got {losses[-1]:.3f}"

def top_guess(sentence):
    return next(iter(fill_in(to_ids(sentence), W, b).values()))[0][0]

for sentence in ["the cat sat on the [MASK]", "the dog sat on the [MASK]", "the [MASK] sat on the mat", "the [MASK] sat on the rug"]:
    best = next(iter(fill_in(to_ids(sentence), W, b).values()))
    print(f"{sentence:27s} -> " + ", ".join(f"{w} {p:.2f}" for w, p in best))
assert top_guess("the cat sat on the [MASK]") == "mat"
assert top_guess("the dog sat on the [MASK]") == "rug"
assert top_guess("the [MASK] sat on the mat") == "cat", "the word after the blank decides: mat goes with cat"
assert top_guess("the [MASK] sat on the rug") == "dog", "the word after the blank decides: rug goes with dog"
print("Part 6 passed")

## Solution

### Part 1: Token and position embeddings

In [ ]:
def embed(ids):
    """One vector per token: its token embedding plus the embedding of its position. Shape (T, d)."""
    return E[ids] + P[np.arange(len(ids))]

### Part 2: One layer of attention in both directions

In [ ]:
def self_attention(X):
    """One layer of attention in both directions.
    Returns (H, weights): H is X plus what every position gathered, shape (T, d),
    and weights is the T x T matrix of attention weights, one row per query position."""
    Q, K, V = X @ W_q, X @ W_k, X @ W_v
    scores = Q @ K.T / np.sqrt(d)
    weights = softmax(scores, axis=1)
    return X + weights @ V, weights

### Part 3: BERT's masking recipe

In [ ]:
def mask_tokens(ids, rng, rate=0.15):
    """BERT's recipe on one sentence. Returns (inputs, labels): inputs is what the model sees,
    labels holds the original id at every chosen position and -100 everywhere else."""
    inputs, labels = ids.copy(), np.full_like(ids, -100)
    chosen = (ids >= 4) & (rng.random(len(ids)) < rate)
    labels[chosen] = ids[chosen]
    roll = rng.random(len(ids))
    inputs[chosen & (roll < 0.8)] = MASK
    swap = chosen & (roll >= 0.8) & (roll < 0.9)
    inputs[swap] = rng.integers(4, n_vocab, swap.sum())
    return inputs, labels

### Part 4: The masked-token loss

In [ ]:
def mlm_loss(logits, labels):
    """Mean cross-entropy over the rows whose label is not -100.
    logits has one row of n_vocab scores per position, labels one id (or -100) per position."""
    m = logits.max(axis=1, keepdims=True)
    logp = logits - m - np.log(np.exp(logits - m).sum(axis=1, keepdims=True))
    keep = labels != -100
    return -logp[keep, labels[keep]].mean()

### Part 5: The prediction at a [MASK]

In [ ]:
def fill_in(inputs, W_out, b_out, top=3):
    """For every [MASK] in inputs, the `top` most likely words and their probabilities.
    Returns {position: [(word, probability), ...]}, most likely first."""
    H, _ = self_attention(embed(inputs))
    probs = softmax(H @ W_out + b_out, axis=1)
    guesses = {}
    for t in np.flatnonzero(inputs == MASK):
        order = np.argsort(probs[t])[::-1][:top]
        guesses[int(t)] = [(vocab[i], float(probs[t, i])) for i in order]
    return guesses

### Part 6: Train the output layer and fill in the blank

In [ ]:
def head_gradient(H, labels, W_out, b_out):
    """Gradient of mlm_loss(H @ W_out + b_out, labels) with respect to W_out and b_out."""
    keep = labels != -100
    dlogits = np.zeros((len(labels), n_vocab))
    dlogits[keep] = softmax(H[keep] @ W_out + b_out, axis=1)
    dlogits[keep, labels[keep]] -= 1
    dlogits /= keep.sum()
    return H.T @ dlogits, dlogits.sum(axis=0)